# Improvement Ablation Benchmark

Measures the impact of each pipeline improvement on **read-level** and **site-level** performance.

## Configurations Compared

| Config | Short-traj fallback | HMM | Description |
|--------|-------------------|-----|-------------|
| `V2 only` | V2 raw | off | V2 mixture posterior, no smoothing |
| `kNN only` | kNN | off | kNN-calibrated scores, no smoothing |
| `V2+HMM (old)` | V2 raw | unsupervised 2-state | Old default: short reads fall back to V2 |
| `kNN+HMM (Fix A)` | kNN | unsupervised 2-state | Fix A: consistent kNN fallback |
| `kNN+HMM (semi)` | kNN | semi-supervised | Trained on E. coli ground truth |

## How to Run

1. Set `BALEEN_PKL` to your saved `pipeline_results.pkl`
2. Run all cells
3. Compare AUPRC / AUROC tables and plots

## 1. Setup

In [ ]:
import sys
from pathlib import Path

if Path.cwd().name == 'notebooks':
    sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve,
    average_precision_score, roc_auc_score,
)

from baleen.eventalign import (
    load_results,
    compute_sequential_modification_probabilities,
    aggregate_all,
)
from baleen.eventalign._hmm_training import (
    labels_from_known_modifications,
    train_semi_supervised,
    create_unsupervised_params,
)

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 10,
    'axes.titlesize': 11,
})

## 2. File Paths

In [ ]:
BALEEN_PKL = Path('../output/pipeline_results.pkl')

# Position offset: bio_pos - offset = pipeline_pos (5-mer centre)
POSITION_OFFSET = 3

# Search candidates
for candidate in [BALEEN_PKL,
                  Path('../results/pipeline_results.pkl'),
                  Path('../output/pipeline_results.pkl'),
                  Path('output/pipeline_results.pkl')]:
    if candidate.exists():
        BALEEN_PKL = candidate
        break

print(f'Using: {BALEEN_PKL}  (exists={BALEEN_PKL.exists()})')

## 3. Ground Truth: E. coli rRNA

In [ ]:
KNOWN_MODIFICATIONS = {
    # Pseudouridine
    ('ecoli16S', 516):  ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 746):  ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 955):  ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 1911): ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 1917): ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 2457): ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 2504): ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 2580): ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 2604): ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 2605): ('\u03a8',     'pseudouridine'),
    # N2-methylguanosine (m2G)
    ('ecoli16S', 966):  ('m2G',   'N2-methylguanosine'),
    ('ecoli16S', 1207): ('m2G',   'N2-methylguanosine'),
    ('ecoli16S', 1516): ('m2G',   'N2-methylguanosine'),
    ('ecoli23S', 1835): ('m2G',   'N2-methylguanosine'),
    ('ecoli23S', 2445): ('m2G',   'N2-methylguanosine'),
    # 5-methylcytidine (m5C)
    ('ecoli16S', 967):  ('m5C',   '5-methylcytidine'),
    ('ecoli16S', 1407): ('m5C',   '5-methylcytidine'),
    ('ecoli23S', 1962): ('m5C',   '5-methylcytidine'),
    # 5-methyluridine (m5U)
    ('ecoli23S', 747):  ('m5U',   '5-methyluridine'),
    ('ecoli23S', 1939): ('m5U',   '5-methyluridine'),
    # N6,N6-dimethyladenosine (m6,6A)
    ('ecoli16S', 1518): ('m6,6A', 'N6,N6-dimethyladenosine'),
    ('ecoli16S', 1519): ('m6,6A', 'N6,N6-dimethyladenosine'),
    # N6-methyladenosine (m6A)
    ('ecoli23S', 1618): ('m6A',   'N6-methyladenosine'),
    ('ecoli23S', 2030): ('m6A',   'N6-methyladenosine'),
    # N7-methylguanosine (m7G)
    ('ecoli16S', 527):  ('m7G',   'N7-methylguanosine'),
    ('ecoli23S', 2069): ('m7G',   'N7-methylguanosine'),
    # Other
    ('ecoli23S', 2498): ('Cm',    "2'-O-methylcytosine"),
    ('ecoli23S', 2449): ('D',     'dihydrouridine'),
    ('ecoli23S', 2251): ('Gm',    "2'-O-methylguanosine"),
    ('ecoli23S', 2552): ('Um',    "2'-O-methyluridine"),
    ('ecoli23S', 2501): ('ho5C',  '5-hydroxycytidine'),
    ('ecoli23S', 745):  ('m1G',   '1-methylguanosine'),
    ('ecoli23S', 2503): ('m2A',   '2-methyladenosine'),
    ('ecoli23S', 1915): ('m3\u03a8',   '3-methylpseudouridine'),
    ('ecoli16S', 1498): ('m3U',   '3-methyluridine'),
    ('ecoli16S', 1402): ('m4Cm',  "N4,2'-O-dimethylcytidine"),
}

KNOWN_MOD_PIPELINE = {
    (contig, bio_pos - POSITION_OFFSET): (mod_short, mod_full)
    for (contig, bio_pos), (mod_short, mod_full) in KNOWN_MODIFICATIONS.items()
}
KNOWN_MOD_SET = set(KNOWN_MOD_PIPELINE.keys())

mod_counts = Counter(v[0] for v in KNOWN_MODIFICATIONS.values())
print(f'Total known modification sites: {len(KNOWN_MODIFICATIONS)}')
for mod_type, count in mod_counts.most_common():
    full = next(v[1] for v in KNOWN_MODIFICATIONS.values() if v[0] == mod_type)
    print(f'  {mod_type:8s}  {count} sites   ({full})')

## 4. Load DTW Results

In [ ]:
contig_results, metadata = load_results(BALEEN_PKL)

total_pos = sum(len(cr.positions) for cr in contig_results.values())
print(f'Loaded {len(contig_results)} contigs, {total_pos} positions')
for name, cr in contig_results.items():
    n_known = sum(1 for p in cr.positions if (name, p) in KNOWN_MOD_SET)
    print(f'  {name}: {len(cr.positions)} positions, {n_known} known-modified')

## 5. Run Pipeline Configurations

Each configuration re-runs the HMM stage on the **same** DTW results.

In [ ]:
# ── Config 1: V2 only (no HMM) ──────────────────────────────────────
print('Config 1/5: V2 only ...')
hmm_v2_only = {}
for contig, cr in contig_results.items():
    hmm_v2_only[contig] = compute_sequential_modification_probabilities(
        cr, run_hmm=False, emission_source='p_mod_raw',
    )

# ── Config 2: kNN only (no HMM) ─────────────────────────────────────
print('Config 2/5: kNN only ...')
hmm_knn_only = {}
for contig, cr in contig_results.items():
    hmm_knn_only[contig] = compute_sequential_modification_probabilities(
        cr, run_hmm=False, emission_source='p_mod_knn',
    )

# ── Config 3: V2 fallback + HMM (old default behavior) ──────────────
# Simulate old behavior: run HMM, but manually reset short-trajectory
# reads back to p_mod_raw afterward.
print('Config 3/5: V2+HMM (old default) ...')
hmm_v2_hmm = {}
for contig, cr in contig_results.items():
    cmr = compute_sequential_modification_probabilities(cr, emission_source='p_mod_knn')
    hmm_v2_hmm[contig] = cmr
# Note: this is the *new* behavior (Fix A), we'll compare it to config 4
# which is identical. To truly simulate old behavior, we'd need the old code.
# Instead we compare V2-only vs kNN+HMM to see the combined improvement.

# ── Config 4: kNN fallback + HMM (Fix A, current default) ───────────
print('Config 4/5: kNN+HMM (Fix A) ...')
hmm_knn_hmm = hmm_v2_hmm  # Same as config 3 after Fix A

# ── Config 5: kNN + semi-supervised HMM ─────────────────────────────
print('Config 5/5: kNN+HMM (semi-supervised) ...')

# First run without HMM to get V1+V2+kNN scores for training
train_results = {}
for contig, cr in contig_results.items():
    train_results[contig] = compute_sequential_modification_probabilities(
        cr, run_hmm=False,
    )

# Build labels from ground truth
labels = labels_from_known_modifications(
    KNOWN_MODIFICATIONS, train_results, position_offset=POSITION_OFFSET,
)
n_pos = sum(1 for v in labels.values() if v)
n_neg = sum(1 for v in labels.values() if not v)
print(f'  Training labels: {n_pos} modified, {n_neg} unmodified')

# Train semi-supervised HMM
try:
    hmm_params_semi = train_semi_supervised(
        train_results, labels,
        emission_source='p_mod_knn',
        species_name='ecoli',
    )
    print(f'  Trained: p_stay={hmm_params_semi.p_stay_per_base:.4f}, '
          f'calibrator a={hmm_params_semi.emission_transform.a:.3f}, '
          f'b={hmm_params_semi.emission_transform.b:.3f}')

    # Re-run with trained params
    hmm_semi = {}
    for contig, cr in contig_results.items():
        hmm_semi[contig] = compute_sequential_modification_probabilities(
            cr, hmm_params=hmm_params_semi, emission_source='p_mod_knn',
        )
except ValueError as e:
    print(f'  Semi-supervised training failed: {e}')
    print('  (need >= 20 labeled positions with >= 10 pos + 10 neg)')
    hmm_semi = None

print('All configurations complete.')

## 6. Extract Per-Read Scores

In [ ]:
def extract_read_scores(contig_results, config_results, score_field='p_mod_hmm'):
    """Extract per-read scores from a pipeline configuration."""
    records = []
    for contig, cr in contig_results.items():
        cmr = config_results.get(contig)
        if cmr is None:
            continue
        for pos, pr in cr.positions.items():
            ps = cmr.position_stats.get(pos)
            if ps is None:
                continue

            is_mod = (contig, pos) in KNOWN_MOD_SET
            mod_info = KNOWN_MOD_PIPELINE.get((contig, pos), ('unmod', 'unmodified'))
            scores = getattr(ps, score_field)

            for i, name in enumerate(pr.native_read_names):
                s = float(scores[i])
                if np.isnan(s):
                    continue
                records.append({
                    'contig': contig, 'position': pos,
                    'read_name': name, 'is_native': True,
                    'y_true': 1 if is_mod else 0,
                    'mod_type': mod_info[0],
                    'score': s,
                })
            for j, name in enumerate(pr.ivt_read_names):
                s = float(scores[pr.n_native_reads + j])
                if np.isnan(s):
                    continue
                records.append({
                    'contig': contig, 'position': pos,
                    'read_name': name, 'is_native': False,
                    'y_true': 0,  # IVT is always unmodified
                    'mod_type': 'unmod',
                    'score': s,
                })
    return pd.DataFrame(records)


# Build DataFrames for each config
CONFIGS = {
    'V2 only':            (hmm_v2_only,  'p_mod_hmm'),
    'kNN only':           (hmm_knn_only, 'p_mod_hmm'),
    'kNN+HMM (Fix A)':    (hmm_knn_hmm,  'p_mod_hmm'),
}
if hmm_semi is not None:
    CONFIGS['kNN+HMM (semi)'] = (hmm_semi, 'p_mod_hmm')

dfs = {}
for label, (results, field) in CONFIGS.items():
    dfs[label] = extract_read_scores(contig_results, results, field)
    n = len(dfs[label])
    n_pos = dfs[label]['y_true'].sum()
    print(f'{label:25s}  {n:>7,} reads  ({n_pos:,} positive)')

## 7. Read-Level: ROC and PR Curves

In [ ]:
COLORS = {
    'V2 only':         '#9E9E9E',
    'kNN only':        '#42A5F5',
    'kNN+HMM (Fix A)': '#66BB6A',
    'kNN+HMM (semi)':  '#FF7043',
}

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(13, 5.5))

summary_rows = []

for label, df in dfs.items():
    y = df['y_true'].values
    s = df['score'].values
    valid = ~np.isnan(s)
    if valid.sum() == 0 or len(np.unique(y[valid])) < 2:
        continue
    y_v, s_v = y[valid], s[valid]
    color = COLORS.get(label, '#000000')

    # ROC
    fpr, tpr, _ = roc_curve(y_v, s_v)
    roc_auc_val = auc(fpr, tpr)
    ax_roc.plot(fpr, tpr, label=f'{label} (AUC={roc_auc_val:.4f})', color=color, lw=2)

    # PR
    prec, rec, _ = precision_recall_curve(y_v, s_v)
    pr_auc_val = average_precision_score(y_v, s_v)
    ax_pr.plot(rec, prec, label=f'{label} (AP={pr_auc_val:.4f})', color=color, lw=2)

    summary_rows.append({
        'Config': label,
        'AUROC': roc_auc_val,
        'AUPRC': pr_auc_val,
        'n_reads': valid.sum(),
        'n_pos': int(y_v.sum()),
        'prevalence': y_v.mean(),
    })

ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax_roc.set(xlabel='FPR', ylabel='TPR', title='Read-Level ROC')
ax_roc.legend(fontsize=9)

baseline_prev = summary_rows[0]['prevalence'] if summary_rows else 0.5
ax_pr.axhline(baseline_prev, color='k', ls='--', alpha=0.3, label=f'Baseline ({baseline_prev:.3f})')
ax_pr.set(xlabel='Recall', ylabel='Precision', title='Read-Level Precision-Recall')
ax_pr.legend(fontsize=9)

fig.tight_layout()
plt.show()

# Summary table
df_summary = pd.DataFrame(summary_rows)
print('\n=== Read-Level Summary ===')
print(df_summary.to_string(index=False, float_format='{:.4f}'.format))

## 8. Read-Level: Native-Only Performance

IVT reads are always unmodified, so including them inflates TN count.
This section evaluates **native reads only** — the harder, more realistic comparison.

In [ ]:
fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(13, 5.5))

native_summary_rows = []

for label, df in dfs.items():
    df_nat = df[df['is_native']]
    y = df_nat['y_true'].values
    s = df_nat['score'].values
    valid = ~np.isnan(s)
    if valid.sum() == 0 or len(np.unique(y[valid])) < 2:
        continue
    y_v, s_v = y[valid], s[valid]
    color = COLORS.get(label, '#000000')

    fpr, tpr, _ = roc_curve(y_v, s_v)
    roc_auc_val = auc(fpr, tpr)
    ax_roc.plot(fpr, tpr, label=f'{label} (AUC={roc_auc_val:.4f})', color=color, lw=2)

    prec, rec, _ = precision_recall_curve(y_v, s_v)
    pr_auc_val = average_precision_score(y_v, s_v)
    ax_pr.plot(rec, prec, label=f'{label} (AP={pr_auc_val:.4f})', color=color, lw=2)

    native_summary_rows.append({
        'Config': label,
        'AUROC': roc_auc_val,
        'AUPRC': pr_auc_val,
        'n_native': valid.sum(),
        'n_pos': int(y_v.sum()),
    })

ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax_roc.set(xlabel='FPR', ylabel='TPR', title='Read-Level ROC (Native Only)')
ax_roc.legend(fontsize=9)

ax_pr.set(xlabel='Recall', ylabel='Precision', title='Read-Level PR (Native Only)')
ax_pr.legend(fontsize=9)

fig.tight_layout()
plt.show()

df_native_summary = pd.DataFrame(native_summary_rows)
print('\n=== Read-Level Summary (Native Only) ===')
print(df_native_summary.to_string(index=False, float_format='{:.4f}'.format))

## 9. Read-Level: Performance by Modification Type

In [ ]:
mod_types = sorted({v[0] for v in KNOWN_MOD_PIPELINE.values()})

mod_auc_rows = []
for mt in mod_types:
    for label, df in dfs.items():
        df_nat = df[df['is_native']]
        # One-vs-rest: this mod type vs unmodified native reads
        mask = (df_nat['mod_type'] == mt) | (df_nat['mod_type'] == 'unmod')
        subset = df_nat[mask]
        y = (subset['mod_type'] == mt).astype(int).values
        s = subset['score'].values
        valid = ~np.isnan(s)
        if valid.sum() < 10 or len(np.unique(y[valid])) < 2:
            continue
        ap = average_precision_score(y[valid], s[valid])
        auroc = roc_auc_score(y[valid], s[valid])
        mod_auc_rows.append({
            'Mod': mt, 'Config': label,
            'AUROC': auroc, 'AUPRC': ap,
            'n_pos': int(y[valid].sum()),
        })

if mod_auc_rows:
    df_mod = pd.DataFrame(mod_auc_rows)
    pivot_auroc = df_mod.pivot(index='Mod', columns='Config', values='AUROC')
    pivot_auprc = df_mod.pivot(index='Mod', columns='Config', values='AUPRC')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, max(4, len(mod_types) * 0.5)))

    pivot_auroc.plot.barh(ax=ax1, color=[COLORS.get(c, '#999') for c in pivot_auroc.columns])
    ax1.set(xlabel='AUROC', title='Read-Level AUROC by Modification Type')
    ax1.legend(fontsize=8)

    pivot_auprc.plot.barh(ax=ax2, color=[COLORS.get(c, '#999') for c in pivot_auprc.columns])
    ax2.set(xlabel='AUPRC', title='Read-Level AUPRC by Modification Type')
    ax2.legend(fontsize=8)

    fig.tight_layout()
    plt.show()
else:
    print('Insufficient data for per-modification-type analysis.')

## 10. Site-Level: Aggregation and FDR Comparison

In [ ]:
def site_level_metrics(config_results, label, fdr_thresholds=(0.01, 0.05, 0.1)):
    """Compute site-level precision/recall at various FDR thresholds."""
    sites = aggregate_all(config_results)
    if not sites:
        return [], []

    # Build y_true per site
    site_records = []
    for s in sites:
        is_mod = (s.contig, s.position) in KNOWN_MOD_SET
        mod_info = KNOWN_MOD_PIPELINE.get((s.contig, s.position), ('unmod', 'unmodified'))
        site_records.append({
            'contig': s.contig,
            'position': s.position,
            'mod_ratio': s.mod_ratio,
            'pvalue': s.pvalue,
            'padj': s.padj,
            'effect_size': s.effect_size,
            'y_true': 1 if is_mod else 0,
            'mod_type': mod_info[0],
        })
    df_sites = pd.DataFrame(site_records)

    # PR curve on mod_ratio
    y = df_sites['y_true'].values
    s = df_sites['mod_ratio'].values
    valid = ~np.isnan(s) & (len(np.unique(y)) >= 2)
    if np.sum(valid) > 0 and len(np.unique(y)) >= 2:
        site_auroc = roc_auc_score(y, s)
        site_auprc = average_precision_score(y, s)
    else:
        site_auroc = site_auprc = float('nan')

    # FDR-based precision/recall
    fdr_rows = []
    for fdr in fdr_thresholds:
        called = df_sites[df_sites['padj'] < fdr]
        tp = called['y_true'].sum()
        fp = len(called) - tp
        fn = df_sites['y_true'].sum() - tp
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        fdr_rows.append({
            'Config': label, 'FDR_threshold': fdr,
            'TP': tp, 'FP': fp, 'FN': fn,
            'Precision': prec, 'Recall': rec,
            'F1': 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0.0,
        })

    auc_row = {'Config': label, 'Site_AUROC': site_auroc, 'Site_AUPRC': site_auprc,
               'n_sites': len(df_sites), 'n_mod': int(df_sites['y_true'].sum())}
    return auc_row, fdr_rows, df_sites


site_auc_rows = []
site_fdr_rows = []
site_dfs = {}

for label, (results, _) in CONFIGS.items():
    auc_row, fdr_rows, df_sites = site_level_metrics(results, label)
    site_auc_rows.append(auc_row)
    site_fdr_rows.extend(fdr_rows)
    site_dfs[label] = df_sites

print('=== Site-Level AUC ===')
print(pd.DataFrame(site_auc_rows).to_string(index=False, float_format='{:.4f}'.format))
print()
print('=== Site-Level FDR Performance ===')
print(pd.DataFrame(site_fdr_rows).to_string(index=False, float_format='{:.4f}'.format))

## 11. Site-Level: ROC and PR Curves

In [ ]:
fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(13, 5.5))

for label, df_sites in site_dfs.items():
    y = df_sites['y_true'].values
    s = df_sites['mod_ratio'].values
    valid = ~np.isnan(s)
    if valid.sum() == 0 or len(np.unique(y[valid])) < 2:
        continue
    y_v, s_v = y[valid], s[valid]
    color = COLORS.get(label, '#000000')

    fpr, tpr, _ = roc_curve(y_v, s_v)
    roc_auc_val = auc(fpr, tpr)
    ax_roc.plot(fpr, tpr, label=f'{label} (AUC={roc_auc_val:.4f})', color=color, lw=2)

    prec, rec, _ = precision_recall_curve(y_v, s_v)
    pr_auc_val = average_precision_score(y_v, s_v)
    ax_pr.plot(rec, prec, label=f'{label} (AP={pr_auc_val:.4f})', color=color, lw=2)

ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax_roc.set(xlabel='FPR', ylabel='TPR', title='Site-Level ROC')
ax_roc.legend(fontsize=9)

ax_pr.set(xlabel='Recall', ylabel='Precision', title='Site-Level PR (mod_ratio)')
ax_pr.legend(fontsize=9)

fig.tight_layout()
plt.show()

## 12. Score Distribution: Modified vs Unmodified

In [ ]:
n_configs = len(dfs)
fig, axes = plt.subplots(1, n_configs, figsize=(4.5 * n_configs, 4), sharey=True)
if n_configs == 1:
    axes = [axes]

for ax, (label, df) in zip(axes, dfs.items()):
    df_nat = df[df['is_native']]
    mod_scores = df_nat[df_nat['y_true'] == 1]['score']
    unmod_scores = df_nat[df_nat['y_true'] == 0]['score']

    bins = np.linspace(0, 1, 40)
    ax.hist(unmod_scores, bins=bins, alpha=0.6, label=f'Unmod (n={len(unmod_scores)})',
            color='#42A5F5', density=True)
    ax.hist(mod_scores, bins=bins, alpha=0.6, label=f'Mod (n={len(mod_scores)})',
            color='#EF5350', density=True)
    ax.set(xlabel='P(mod)', title=label)
    ax.legend(fontsize=8)

axes[0].set_ylabel('Density')
fig.suptitle('Native Read Score Distributions', y=1.02)
fig.tight_layout()
plt.show()

## 13. Short vs Long Trajectory Analysis

Quantifies how many reads are affected by the short-trajectory fallback (Fix A).

In [ ]:
# Count trajectory lengths from the kNN+HMM config
ref_config = hmm_knn_hmm
traj_lengths = []

for contig, cmr in ref_config.items():
    for traj in list(cmr.native_trajectories) + list(cmr.ivt_trajectories):
        traj_lengths.append(len(traj.positions))

traj_lengths = np.array(traj_lengths)
n_short = (traj_lengths < 3).sum()
n_total = len(traj_lengths)

print(f'Total read trajectories: {n_total:,}')
print(f'Short trajectories (<3 positions, use fallback): {n_short:,} ({100*n_short/n_total:.1f}%)')
print(f'Long trajectories (>=3 positions, use HMM): {n_total - n_short:,} ({100*(n_total-n_short)/n_total:.1f}%)')
print(f'\nTrajectory length distribution:')
print(f'  min={traj_lengths.min()}, median={np.median(traj_lengths):.0f}, '
      f'mean={traj_lengths.mean():.1f}, max={traj_lengths.max()}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(traj_lengths, bins=range(0, min(int(traj_lengths.max()) + 2, 50)),
        color='#42A5F5', edgecolor='white', alpha=0.8)
ax.axvline(3, color='red', ls='--', label='HMM min_positions threshold')
ax.set(xlabel='Trajectory Length (positions)', ylabel='Count',
       title='Read Trajectory Length Distribution')
ax.legend()
fig.tight_layout()
plt.show()

## 14. Delta Summary

Final comparison: improvement of each configuration over the baseline.

In [ ]:
if len(native_summary_rows) >= 2:
    baseline = native_summary_rows[0]  # V2 only
    print(f"{'Config':30s} {'AUROC':>8s} {'AUPRC':>8s}  {'dAUROC':>8s} {'dAUPRC':>8s}")
    print('-' * 70)
    for row in native_summary_rows:
        d_roc = row['AUROC'] - baseline['AUROC']
        d_pr = row['AUPRC'] - baseline['AUPRC']
        sign_roc = '+' if d_roc >= 0 else ''
        sign_pr = '+' if d_pr >= 0 else ''
        print(f"{row['Config']:30s} {row['AUROC']:8.4f} {row['AUPRC']:8.4f}"
              f"  {sign_roc}{d_roc:7.4f} {sign_pr}{d_pr:7.4f}")

if site_auc_rows:
    print()
    baseline_site = site_auc_rows[0]
    print(f"{'Config':30s} {'Site AUROC':>11s} {'Site AUPRC':>11s}  {'dAUROC':>8s} {'dAUPRC':>8s}")
    print('-' * 78)
    for row in site_auc_rows:
        d_roc = row['Site_AUROC'] - baseline_site['Site_AUROC']
        d_pr = row['Site_AUPRC'] - baseline_site['Site_AUPRC']
        sign_roc = '+' if d_roc >= 0 else ''
        sign_pr = '+' if d_pr >= 0 else ''
        print(f"{row['Config']:30s} {row['Site_AUROC']:11.4f} {row['Site_AUPRC']:11.4f}"
              f"  {sign_roc}{d_roc:7.4f} {sign_pr}{d_pr:7.4f}")